In [1]:
import pandas as pd
import numpy as np
from itertools import combinations

In [2]:
df = pd.read_csv("../data/preprocessed_data.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (100000, 18)


,CustomerID,ProductID,Quantity,Price,TransactionDate,PaymentMethod,StoreLocation,ProductCategory,DiscountApplied(%),TotalAmount,Year,Month,Day,Week,Quarter,DayOfWeek,Weekend,Season
0,109318,C,7,80.079844,2025-12-26 12:32:00,Cash,"176 Andrew Cliffs\nBaileyfort, HI 93354",Books,18.677100,455.862764,2025,12,26,52,4,Friday,False,Winter
1,993229,C,4,75.195229,2025-08-05 00:00:00,Cash,"11635 William Well Suite 809\nEast Kara, MT 19483",Home Decor,14.121365,258.306546,2025,8,5,32,3,Tuesday,False,Monsoon
2,579675,A,8,31.528816,2026-03-11 18:51:00,Cash,"910 Mendez Ville Suite 909\nPort Lauraland, MO...",Books,15.943701,212.015651,2026,3,11,11,1,Wednesday,False,Summer
3,799826,D,5,98.880218,2025-10-27 22:00:00,PayPal,"87522 Sharon Corners Suite 500\nLake Tammy, MO...",Books,6.686337,461.343769,2025,10,27,44,4,Monday,False,Autumn
4,121413,A,7,93.188512,2025-12-22 11:38:00,Cash,"0070 Michelle Island Suite 143\nHoland, VA 80142",Electronics,4.030096,626.030484,2025,12,22,52,4,Monday,False,Winter


In [3]:
print("Unique Products:")
print(df["ProductID"].unique())

print("\nProduct Purchase Counts:")
print(df["ProductID"].value_counts())

Unique Products:
<StringArray>
['C', 'A', 'D', 'B']
Length: 4, dtype: str

Product Purchase Counts:
ProductID
C    25209
D    25042
B    24997
A    24752
Name: count, dtype: int64


In [4]:
# Group products purchased by each customer transaction
transaction_products = (
    df.groupby(["CustomerID", "TransactionDate"])["ProductID"]
    .apply(lambda x: sorted(set(x)))
)

print("Total Transactions:", len(transaction_products))
transaction_products.head(10)

Total Transactions: 100000


CustomerID  TransactionDate    
14          2025-08-06 06:45:00    [A]
42          2025-05-19 21:52:00    [B]
49          2025-06-05 13:10:00    [A]
59          2025-08-19 03:50:00    [B]
            2026-04-01 01:06:00    [D]
65          2025-06-18 10:29:00    [D]
87          2025-08-21 15:23:00    [D]
96          2026-04-06 12:35:00    [D]
98          2025-12-20 11:49:00    [D]
100         2025-05-13 17:17:00    [A]
Name: ProductID, dtype: object

In [5]:
# Create a basket of unique products purchased by each customer

customer_baskets = (
    df.groupby("CustomerID")["ProductID"]
    .apply(lambda x: sorted(set(x)))
)

print("Total Customers:", len(customer_baskets))
customer_baskets.head(10)

Total Customers: 95215


CustomerID
14        [A]
42        [B]
49        [A]
59     [B, D]
65        [D]
87        [D]
96        [D]
98        [D]
100       [A]
101       [C]
Name: ProductID, dtype: object

In [6]:
# Check how many different products each customer purchased

basket_sizes = customer_baskets.apply(len)

print("Average Products per Customer:", basket_sizes.mean())
print("Maximum Products per Customer :", basket_sizes.max())

print("\nBasket Size Distribution:")
print(basket_sizes.value_counts().sort_index())

Average Products per Customer: 1.0372420311925643
Maximum Products per Customer : 3

Basket Size Distribution:
ProductID
1    91731
2     3422
3       62
Name: count, dtype: int64


In [7]:
# Generate unique product pairs for customers
# who purchased more than one product

product_pairs = []

for products in customer_baskets:
    if len(products) > 1:
        pairs = combinations(products, 2)
        product_pairs.extend(pairs)

print("Total Product Pairs:", len(product_pairs))

product_pairs[:20]

Total Product Pairs: 3608


[('B', 'D'),
 ('A', 'D'),
 ('B', 'C'),
 ('B', 'D'),
 ('C', 'D'),
 ('A', 'B'),
 ('A', 'B'),
 ('A', 'B'),
 ('C', 'D'),
 ('A', 'D'),
 ('A', 'C'),
 ('B', 'D'),
 ('A', 'C'),
 ('A', 'C'),
 ('B', 'C'),
 ('A', 'C'),
 ('A', 'D'),
 ('B', 'D'),
 ('A', 'C'),
 ('B', 'D')]

In [8]:
# Count how many customers purchased each product pair

pair_counts = (
    pd.Series(product_pairs)
    .value_counts()
    .reset_index()
)

pair_counts.columns = ["ProductPair", "CoPurchaseCount"]

print("Unique Product Pairs:", len(pair_counts))

pair_counts

Unique Product Pairs: 6


,ProductPair,CoPurchaseCount
0,"(A, B)",645
1,"(C, D)",621
2,"(B, D)",613
3,"(A, D)",605
4,"(B, C)",563
5,"(A, C)",561


In [9]:
# Total customers who purchased each individual product

product_customer_counts = (
    df.groupby("ProductID")["CustomerID"]
    .nunique()
)

# Calculate support for each product pair

pair_counts["Product1"] = pair_counts["ProductPair"].apply(lambda x: x[0])
pair_counts["Product2"] = pair_counts["ProductPair"].apply(lambda x: x[1])

pair_counts["Product1Count"] = pair_counts["Product1"].map(
    product_customer_counts
)

pair_counts["Product2Count"] = pair_counts["Product2"].map(
    product_customer_counts
)

pair_counts["Support_Product1"] = (
    pair_counts["CoPurchaseCount"] /
    pair_counts["Product1Count"]
)

pair_counts["Support_Product2"] = (
    pair_counts["CoPurchaseCount"] /
    pair_counts["Product2Count"]
)

pair_counts["AverageSupport"] = (
    pair_counts["Support_Product1"] +
    pair_counts["Support_Product2"]
) / 2

pair_counts = pair_counts.sort_values(
    "AverageSupport",
    ascending=False
).reset_index(drop=True)

pair_counts.round(4)

,ProductPair,CoPurchaseCount,Product1,Product2,Product1Count,Product2Count,Support_Product1,Support_Product2,AverageSupport
0,"(A, B)",645,A,B,24444,24705,0.0264,0.0261,0.0262
1,"(C, D)",621,C,D,24897,24715,0.0249,0.0251,0.0250
2,"(B, D)",613,B,D,24705,24715,0.0248,0.0248,0.0248
3,"(A, D)",605,A,D,24444,24715,0.0248,0.0245,0.0246
4,"(A, C)",561,A,C,24444,24897,0.0230,0.0225,0.0227
5,"(B, C)",563,B,C,24705,24897,0.0228,0.0226,0.0227


In [10]:
print("="*60)
print("PRODUCT CO-PURCHASE ANALYSIS")
print("="*60)

print("\nProduct Pair Statistics:")
print(
    pair_counts[
        [
            "ProductPair",
            "CoPurchaseCount",
            "AverageSupport"
        ]
    ].round(4)
)

PRODUCT CO-PURCHASE ANALYSIS

Product Pair Statistics:
  ProductPair  CoPurchaseCount  AverageSupport
0      (A, B)              645          0.0262
1      (C, D)              621          0.0250
2      (B, D)              613          0.0248
3      (A, D)              605          0.0246
4      (A, C)              561          0.0227
5      (B, C)              563          0.0227


In [11]:
# Improved Recommendation Engine
# Uses co-purchase frequency and support

def get_improved_recommendations(product_id, top_n=3):

    recommendations = []

    for _, row in pair_counts.iterrows():

        if row["Product1"] == product_id:
            recommendations.append({
                "ProductID": row["Product2"],
                "CoPurchaseCount": row["CoPurchaseCount"],
                "Support": row["Support_Product1"]
            })

        elif row["Product2"] == product_id:
            recommendations.append({
                "ProductID": row["Product1"],
                "CoPurchaseCount": row["CoPurchaseCount"],
                "Support": row["Support_Product2"]
            })

    recommendations_df = pd.DataFrame(recommendations)

    if recommendations_df.empty:
        return recommendations_df

    return recommendations_df.sort_values(
        by=["CoPurchaseCount", "Support"],
        ascending=False
    ).head(top_n).reset_index(drop=True)

In [12]:
# Test recommendations for each product

for product in sorted(df["ProductID"].unique()):

    print("="*50)
    print(f"Recommendations for Product {product}")
    print("="*50)

    print(
        get_improved_recommendations(product)
    )

Recommendations for Product A
  ProductID  CoPurchaseCount   Support
0         B              645  0.026387
1         D              605  0.024750
2         C              561  0.022950
Recommendations for Product B
  ProductID  CoPurchaseCount   Support
0         A              645  0.026108
1         D              613  0.024813
2         C              563  0.022789
Recommendations for Product C
  ProductID  CoPurchaseCount   Support
0         D              621  0.024943
1         B              563  0.022613
2         A              561  0.022533
Recommendations for Product D
  ProductID  CoPurchaseCount   Support
0         C              621  0.025126
1         B              613  0.024803
2         A              605  0.024479


In [13]:
# Milestone 2 baseline: popularity-based recommendations

product_popularity = (
    df["ProductID"]
    .value_counts()
    .reset_index()
)

product_popularity.columns = [
    "ProductID",
    "PurchaseCount"
]

def get_baseline_recommendations(product_id, top_n=3):
    return (
        product_popularity[
            product_popularity["ProductID"] != product_id
        ]
        .head(top_n)
        .reset_index(drop=True)
    )

In [14]:
# Compare both recommendation methods

for product in sorted(df["ProductID"].unique()):

    baseline = get_baseline_recommendations(product)
    improved = get_improved_recommendations(product)

    print("="*60)
    print(f"Product: {product}")
    print("="*60)

    print("Baseline Recommendations:")
    print(baseline["ProductID"].tolist())

    print("\nImproved Recommendations:")
    print(improved["ProductID"].tolist())

    print()

Product: A
Baseline Recommendations:
['C', 'D', 'B']

Improved Recommendations:
['B', 'D', 'C']

Product: B
Baseline Recommendations:
['C', 'D', 'A']

Improved Recommendations:
['A', 'D', 'C']

Product: C
Baseline Recommendations:
['D', 'B', 'A']

Improved Recommendations:
['D', 'B', 'A']

Product: D
Baseline Recommendations:
['C', 'B', 'A']

Improved Recommendations:
['C', 'B', 'A']



In [15]:
# Compare the strength of recommendations

validation_results = []

for product in sorted(df["ProductID"].unique()):

    improved = get_improved_recommendations(product)

    avg_copurchase = improved["CoPurchaseCount"].mean()
    avg_support = improved["Support"].mean()

    validation_results.append({
        "Product": product,
        "AverageCoPurchaseCount": avg_copurchase,
        "AverageSupport": avg_support
    })

validation_df = pd.DataFrame(validation_results)

print("="*60)
print("RECOMMENDATION VALIDATION")
print("="*60)

print(validation_df.round(4))

RECOMMENDATION VALIDATION
  Product  AverageCoPurchaseCount  AverageSupport
0       A                603.6667          0.0247
1       B                607.0000          0.0246
2       C                581.6667          0.0234
3       D                613.0000          0.0248


In [16]:
# Compare the co-purchase strength of the top recommendation

comparison = []

for product in sorted(df["ProductID"].unique()):

    baseline = get_baseline_recommendations(product, top_n=1)
    improved = get_improved_recommendations(product, top_n=1)

    baseline_product = baseline.iloc[0]["ProductID"]
    improved_product = improved.iloc[0]["ProductID"]

    # Find co-purchase counts
    baseline_pair = pair_counts[
        (
            ((pair_counts["Product1"] == product) &
             (pair_counts["Product2"] == baseline_product))
            |
            ((pair_counts["Product1"] == baseline_product) &
             (pair_counts["Product2"] == product))
        )
    ]

    improved_pair = pair_counts[
        (
            ((pair_counts["Product1"] == product) &
             (pair_counts["Product2"] == improved_product))
            |
            ((pair_counts["Product1"] == improved_product) &
             (pair_counts["Product2"] == product))
        )
    ]

    comparison.append({
        "Product": product,
        "BaselineTopRecommendation": baseline_product,
        "BaselineCoPurchase": baseline_pair.iloc[0]["CoPurchaseCount"],
        "ImprovedTopRecommendation": improved_product,
        "ImprovedCoPurchase": improved_pair.iloc[0]["CoPurchaseCount"]
    })

comparison_df = pd.DataFrame(comparison)

print("="*60)
print("BASELINE VS IMPROVED TOP-1 VALIDATION")
print("="*60)

print(comparison_df)

print("\nAverage Baseline Top-1 Co-Purchase:",
      comparison_df["BaselineCoPurchase"].mean())

print("Average Improved Top-1 Co-Purchase:",
      comparison_df["ImprovedCoPurchase"].mean())

BASELINE VS IMPROVED TOP-1 VALIDATION
  Product BaselineTopRecommendation  BaselineCoPurchase  \
0       A                         C                 561   
1       B                         C                 563   
2       C                         D                 621   
3       D                         C                 621   

  ImprovedTopRecommendation  ImprovedCoPurchase  
0                         B                 645  
1                         A                 645  
2                         D                 621  
3                         C                 621  

Average Baseline Top-1 Co-Purchase: 591.5
Average Improved Top-1 Co-Purchase: 633.0


In [17]:
# Final Improved Recommendation Output

recommendation_rows = []

for product in sorted(df["ProductID"].unique()):

    recommendations = get_improved_recommendations(product, top_n=3)

    for rank, row in recommendations.iterrows():
        recommendation_rows.append({
            "ProductID": product,
            "RecommendationRank": rank + 1,
            "RecommendedProduct": row["ProductID"],
            "CoPurchaseCount": row["CoPurchaseCount"],
            "Support": row["Support"]
        })

recommendation_output = pd.DataFrame(recommendation_rows)

print("="*60)
print("FINAL RECOMMENDATION OUTPUT")
print("="*60)

print(recommendation_output)

FINAL RECOMMENDATION OUTPUT
   ProductID  RecommendationRank RecommendedProduct  CoPurchaseCount   Support
0          A                   1                  B              645  0.026387
1          A                   2                  D              605  0.024750
2          A                   3                  C              561  0.022950
3          B                   1                  A              645  0.026108
4          B                   2                  D              613  0.024813
5          B                   3                  C              563  0.022789
6          C                   1                  D              621  0.024943
7          C                   2                  B              563  0.022613
8          C                   3                  A              561  0.022533
9          D                   1                  C              621  0.025126
10         D                   2                  B              613  0.024803
11         D            

In [18]:
# Save improved recommendation output

recommendation_output.to_csv(
    "../data/improved_product_recommendations.csv",
    index=False
)

print("Recommendation CSV saved successfully.")
print("../data/improved_product_recommendations.csv")

Recommendation CSV saved successfully.
../data/improved_product_recommendations.csv


In [19]:
import joblib

joblib.dump(
    pair_counts,
    "../models/improved_product_recommendations.pkl"
)

print("="*60)
print("Recommendation Model Data Saved")
print("="*60)
print("../models/improved_product_recommendations.pkl")

Recommendation Model Data Saved
../models/improved_product_recommendations.pkl


In [20]:
print("="*60)
print("DAY 7 - RECOMMENDATION ENGINE SUMMARY")
print("="*60)

print("\nBaseline Method:")
print("Popularity-based recommendation")

print("\nImproved Method:")
print("Customer-level co-purchase recommendation")

print("\nValidation:")
print(f"Baseline Average Top-1 Co-Purchase : "
      f"{comparison_df['BaselineCoPurchase'].mean():.1f}")

print(f"Improved Average Top-1 Co-Purchase : "
      f"{comparison_df['ImprovedCoPurchase'].mean():.1f}")

improvement = (
    (comparison_df["ImprovedCoPurchase"].mean()
     - comparison_df["BaselineCoPurchase"].mean())
    / comparison_df["BaselineCoPurchase"].mean()
) * 100

print(f"Relative Improvement               : {improvement:.2f}%")

print("\nOutput Files:")
print("../data/improved_product_recommendations.csv")
print("../models/improved_product_recommendations.pkl")

DAY 7 - RECOMMENDATION ENGINE SUMMARY

Baseline Method:
Popularity-based recommendation

Improved Method:
Customer-level co-purchase recommendation

Validation:
Baseline Average Top-1 Co-Purchase : 591.5
Improved Average Top-1 Co-Purchase : 633.0
Relative Improvement               : 7.02%

Output Files:
../data/improved_product_recommendations.csv
../models/improved_product_recommendations.pkl
